### 1.1 方式一： 直接调用

In [1]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """

    return city + "晴天，温度 15℃"

# 调用工具
get_weather.invoke({
    "city":"北京"
})

'北京晴天，温度 15℃'

### 1.2 基于模型调用

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from config.config import DeepSeekModel

model = init_chat_model(
    base_url=DeepSeekModel.deepseek_base_url,
    api_key=DeepSeekModel.deepseek_api_key,
    model=DeepSeekModel.deepseek_model,
    model_provider="openai"
)

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """

    return city + "晴天，温度 15℃"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# ai 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")

# 检查 ai 是否要调用工具
if response.tool_calls:
    print("ai想调用工具：",response.tool_calls)
else:
    print("ai直接回答",response.content)


ai想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_rDiBePe8FfKXajFrG2q29232', 'type': 'tool_call'}]


# 2 从Message流转查看工具的调用

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from config.config import DeepSeekModel
from langchain_core.messages import HumanMessage


model = init_chat_model(
    base_url=DeepSeekModel.deepseek_base_url,
    api_key=DeepSeekModel.deepseek_api_key,
    model=DeepSeekModel.deepseek_model,
    model_provider="openai"
)

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """

    return city + "晴天，温度 15℃"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# 声明消息列表
message = [
    HumanMessage("北京天气如何")
]

# ai 可以决定是否调用工具
response = model_with_tools.invoke(message)

# 添加aimessage到列表中
message.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        message.append(tool_response)

print("================> message <=====================")
for msg in message:
    msg.pretty_print()
print("================> message <=====================")
final_response = model_with_tools.invoke(message)
print(f"final_response: \n{final_response}")


message.clear()

<class 'langchain_core.messages.tool.ToolMessage'>
================> message <=====================
================================ Human Message =================================

北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_TLbQVhm0QPnLUKhGMR6M2275)
 Call ID: call_00_TLbQVhm0QPnLUKhGMR6M2275
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京晴天，温度 15℃
================> message <=====================
final_response: 
content='北京现在是晴天，气温 15℃，天气不错，适合外出活动。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 379, 'total_tokens': 395, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128,